# Week 03 - Tom and Jerry OOP

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |
| 시트 기준 열 | Tom and Jerry Show |

이 노트북은 과제 요구사항을 학습용으로 재구성한 설명형 산출물이다. 원본 코드를 그대로 복사하지 않고, 같은 개념을 작은 로컬 예제로 다시 구현한다.

## 목표

클래스, 속성, 메서드, 객체 상호작용을 Tom and Jerry 시나리오로 구현한다.

모든 코드는 외부 서비스 접속 없이 실행되도록 구성했다. 실제 API나 웹사이트를 사용할 때는 같은 처리 흐름에서 데이터 입력 부분만 교체하면 된다.

## 1. 클래스 설계

객체지향 프로그래밍에서는 데이터를 속성으로 저장하고, 행동을 메서드로 묶는다. Tom과 Jerry를 캐릭터 객체로 만들면 추적, 회피, 점수 변화 같은 상호작용을 표현할 수 있다.

In [1]:
from dataclasses import dataclass, field
import random
import pandas as pd

random.seed(2151050)

@dataclass
class Character:
    name: str
    role: str
    energy: int = 100
    score: int = 0
    history: list[str] = field(default_factory=list)

    def act(self, action: str, target: "Character") -> dict:
        if action == "chase":
            delta = random.randint(5, 12)
            self.energy -= delta
            target.energy -= max(delta - 4, 1)
            self.score += 2
            message = f"{self.name} chased {target.name}"
        elif action == "escape":
            delta = random.randint(3, 9)
            self.energy -= delta
            self.score += 3
            target.score -= 1
            message = f"{self.name} escaped from {target.name}"
        else:
            self.energy -= 1
            message = f"{self.name} waited"

        self.energy = max(self.energy, 0)
        target.energy = max(target.energy, 0)
        self.history.append(message)
        return {"actor": self.name, "target": target.name, "action": action, "message": message}

    def status(self) -> dict:
        return {"name": self.name, "role": self.role, "energy": self.energy, "score": self.score}

tom = Character("Tom", "cat")
jerry = Character("Jerry", "mouse")
tom, jerry

(Character(name='Tom', role='cat', energy=100, score=0, history=[]),
 Character(name='Jerry', role='mouse', energy=100, score=0, history=[]))

## 2. 객체 상호작용 실행

같은 클래스에서 만든 객체라도 각자 독립적인 상태를 가진다. 아래 반복문은 행동 로그를 만들고, 객체의 에너지와 점수를 계속 갱신한다.

In [2]:
actions = [
    tom.act("chase", jerry),
    jerry.act("escape", tom),
    tom.act("chase", jerry),
    jerry.act("escape", tom),
    tom.act("wait", jerry),
]

log_df = pd.DataFrame(actions)
status_df = pd.DataFrame([tom.status(), jerry.status()])

display(log_df)
display(status_df)

,actor,target,action,message
0,Tom,Jerry,chase,Tom chased Jerry
1,Jerry,Tom,escape,Jerry escaped from Tom
2,Tom,Jerry,chase,Tom chased Jerry
3,Jerry,Tom,escape,Jerry escaped from Tom
4,Tom,Jerry,wait,Tom waited


,name,role,energy,score
0,Tom,cat,77,2
1,Jerry,mouse,74,6


## 3. 왜 이렇게 구현하는가

`Character` 클래스는 캐릭터 공통 속성과 행동을 한 곳에 모은다. 새 캐릭터를 추가해도 같은 `act`, `status` 메서드를 재사용할 수 있어 코드 중복이 줄어든다.

In [3]:
spike = Character("Spike", "dog", energy=120)
spike.act("chase", tom)
all_status = pd.DataFrame([tom.status(), jerry.status(), spike.status()])

assert set(all_status["name"]) == {"Tom", "Jerry", "Spike"}
assert all_status["energy"].between(0, 120).all()
assert len(tom.history) >= 1

all_status

,name,role,energy,score
0,Tom,cat,70,2
1,Jerry,mouse,74,6
2,Spike,dog,109,2
